In [ ]:
import fastf1
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import os

def get_team_color_hex(driver_code):
    # Accurate 2025 Constructor Colors
    colors = {
        'VER': '#3671C6', 'LAW': '#3671C6', # Red Bull
        'NOR': '#FF8000', 'PIA': '#FF8000', # McLaren
        'LEC': '#E80020', 'HAM': '#E80020', # Ferrari
        'RUS': '#27F4D2', 'ANT': '#27F4D2', # Mercedes
        'ALO': '#2293D1', 'STR': '#2293D1', # Aston Martin
        'SAI': '#66C2FF', 'ALB': '#66C2FF', # Williams
        'GAS': '#0090FF', 'DOO': '#0090FF', # Alpine
        'OCO': '#FFFFFF', 'BEA': '#FFFFFF', # Haas
        'TSU': '#66CCFF', 'HAD': '#66CCFF', # Racing Bulls
        'HUL': '#00E600', 'BOR': '#00E600'  # Sauber
    }
    return colors.get(driver_code, '#888888')

def get_tire_info(compound):
    compound = str(compound).upper()
    if 'SOFT' in compound: return '#FF3333', 'S'
    if 'MEDIUM' in compound: return '#FFFF00', 'M'
    if 'HARD' in compound: return '#FFFFFF', 'H'
    if 'INTER' in compound: return '#00FF00', 'I'
    if 'WET' in compound: return '#0000FF', 'W'
    return '#888888', '?'

def hex_to_rgb(hex_col):
    hex_col = hex_col.lstrip('#')
    return tuple(int(hex_col[i:i+2], 16) for i in (0, 2, 4))

def get_ordinal(n):
    if 11 <= (n % 100) <= 13: return f"{n}th"
    return f"{n}" + ['th', 'st', 'nd', 'rd', 'th'][min(n % 10, 4)]

def load_fonts(base_size):
    bold = "Formula1-Bold_web_0.ttf.ttf"
    reg = "Formula1-Regular_web_0.ttf.ttf"
    
    if not os.path.exists(bold) or not os.path.exists(reg):
        print("Warning: F1 fonts not found, falling back to default.")
        d = ImageFont.load_default()
        return d, d, d, d, d

    title_font = ImageFont.truetype(bold, int(base_size * 1.5))
    axis_font = ImageFont.truetype(bold, int(base_size * 0.8))
    label_font = ImageFont.truetype(bold, int(base_size * 1.1))
    tire_font = ImageFont.truetype(bold, int(base_size * 1.0)) # Increased tire letter
    gap_font = ImageFont.truetype(reg, int(base_size * 0.75))
    return title_font, axis_font, label_font, tire_font, gap_font

def create_race_timelapse(year=2025, gp='Australia', session_type='R', 
                          save_path='output.mp4', duration=30, fps=60, portrait=True):
    
    print(f"Loading pure time data for {year} {gp} Session: {session_type}...")
    session = fastf1.get_session(year, gp, session_type)
    session.load(laps=True, telemetry=False)
    
    laps_data = session.laps
    drivers = pd.unique(laps_data['Driver'])
    total_laps = int(laps_data['LapNumber'].max())
    
    d_data = {}
    global_start_time = float('inf')
    global_end_time = 0.0
    
    # PURE TIME DATA EXTRACTION (NO FAKE MATH)
    for drv in drivers:
        drv_laps = laps_data.pick_driver(drv).dropna(subset=['LapNumber', 'Position', 'Time', 'LapStartTime'])
        if drv_laps.empty: continue
            
        lap_ends = drv_laps['Time'].dt.total_seconds().values
        lap_starts = drv_laps['LapStartTime'].dt.total_seconds().values
        
        start_time = lap_starts[0]
        times = np.insert(lap_ends, 0, start_time)
        laps = np.insert(drv_laps['LapNumber'].values, 0, 0)
        pos = np.insert(drv_laps['Position'].values, 0, drv_laps['Position'].values[0])
        compounds = np.insert(drv_laps['Compound'].values, 0, drv_laps['Compound'].values[0])
        
        global_start_time = min(global_start_time, start_time)
        global_end_time = max(global_end_time, lap_ends[-1])
        
        d_data[drv] = {
            'laps': laps, 'pos': pos, 'times': times, 'compounds': compounds,
            'color': get_team_color_hex(drv),
            'max_time': lap_ends[-1]
        }

    canvas_w, canvas_h = (1080, 1920) if portrait else (1920, 1080)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(save_path, fourcc, fps, (canvas_w, canvas_h))
    
    base_size = (canvas_w if portrait else canvas_h) * 0.025
    title_f, axis_f, label_f, tire_f, gap_f = load_fonts(base_size)
    
    # 0.9 SCALE NLE LAYOUT
    box_w = canvas_w * 0.9
    box_h = canvas_h * 0.9
    box_x = (canvas_w - box_w) / 2
    box_y = (canvas_h - box_h) / 2
    
    # Inner Margins to prevent graph clipping
    graph_offset_l = 140 # Space for 1st, 2nd...
    graph_offset_t = 160 # Space for Title
    graph_offset_b = 100 # Space for LAP X-axis
    graph_offset_r = 50
    
    graph_w = box_w - graph_offset_l - graph_offset_r
    graph_h = box_h - graph_offset_t - graph_offset_b
    
    # We restrict the DOT plotting width so names have room to render on the right without clipping
    plot_w = graph_w - 200 
    
    # DOUBLED THICKNESSES
    line_w = int(canvas_w * 0.012)
    dot_r = int(canvas_w * 0.018)
    
    total_frames = duration * fps
    print(f"Rendering {total_frames} frames to {save_path}...")
    
    for frame_idx in range(total_frames):
        progress = frame_idx / (total_frames - 1)
        current_time = global_start_time + progress * (global_end_time - global_start_time)
        
        current_states = {}
        leader_drv = max(d_data.keys(), key=lambda d: np.interp(min(current_time, d_data[d]['max_time']), d_data[d]['times'], d_data[d]['laps']))
        leader_data = d_data[leader_drv]
        
        for drv, data in d_data.items():
            # EXACT REAL PHYSICS
            eval_time = min(current_time, data['max_time'])
            cur_lap = np.interp(eval_time, data['times'], data['laps'])
            cur_pos = np.interp(eval_time, data['times'], data['pos'])
            
            leader_t_at_lap = np.interp(cur_lap, leader_data['laps'], leader_data['times'])
            gap_to_leader = max(0.0, eval_time - leader_t_at_lap)
            
            is_dnf = data['laps'][-1] < total_laps
            time_since_finish = current_time - data['max_time']
            alpha = max(0, int(255 - (time_since_finish * 5))) if (is_dnf and time_since_finish > 0) else 255
            
            current_states[drv] = {
                'lap': cur_lap, 'pos': cur_pos, 'eval_time': eval_time,
                'alpha': alpha, 'gap': gap_to_leader
            }

        # Auto-Zoom
        active_laps = [s['lap'] for d, s in current_states.items() if s['alpha'] > 0]
        leader_lap = current_states[leader_drv]['lap']
        last_lap = min(active_laps) if active_laps else leader_lap
        
        window_size = max(1.0, (leader_lap - last_lap) * 1.15)
        view_max_x = leader_lap + (window_size * 0.1)
        view_min_x = view_max_x - window_size

        def get_x(lap_num): return ((lap_num - view_min_x) / window_size) * plot_w
        # Leave padding inside the graph height so top/bottom dots don't clip bounds
        def get_y(position): return dot_r + 10 + ((position - 1) / 19.0) * (graph_h - 2 * (dot_r + 10))

        # --- 1. BASE LAYER ---
        base_img = Image.new('RGBA', (canvas_w, canvas_h), (0, 0, 0, 255))
        draw_base = ImageDraw.Draw(base_img)
        draw_base.rectangle([box_x, box_y, box_x + box_w, box_y + box_h], outline=(255, 255, 255, 255), width=3)
        
        # Y-Axis Labels (Drawn on base so lines can't overlap them)
        for pos in range(1, 21):
            y_abs = box_y + graph_offset_t + get_y(pos)
            pos_str = get_ordinal(pos)
            try:
                bbox = axis_f.getbbox(pos_str)
                draw_base.text((box_x + graph_offset_l - (bbox[2]-bbox[0]) - 20, y_abs - (bbox[3]-bbox[1])/2), pos_str, font=axis_f, fill=(200, 200, 200, 255))
            except AttributeError: pass

        # Absolute X-Axis Ticks
        tick_step = 5.0 if window_size > 10 else 1.0
        first_tick = int(view_min_x / tick_step) * tick_step
        for tick_idx in range(int(window_size / tick_step) + 3):
            tick = first_tick + (tick_idx * tick_step)
            if tick < 0: continue
            x_rel = get_x(tick)
            if 0 <= x_rel <= plot_w:
                x_abs = box_x + graph_offset_l + x_rel
                draw_base.line([(x_abs, box_y + graph_offset_t), (x_abs, box_y + graph_offset_t + graph_h)], fill=(30, 30, 30, 255), width=1)
                tick_str = str(int(tick))
                try:
                    bbox = axis_f.getbbox(tick_str)
                    draw_base.text((x_abs - (bbox[2]-bbox[0])/2, box_y + graph_offset_t + graph_h + 15), tick_str, font=axis_f, fill=(200, 200, 200, 255))
                except AttributeError: pass

        # Title Auto-Scaling
        title_str = f"{gp.upper()} {year} - RACE - LAP {int(leader_lap)}/{total_laps}"
        cur_title_f = title_f
        cur_size = int(base_size * 1.5)
        try:
            t_bbox = cur_title_f.getbbox(title_str)
            while (t_bbox[2] - t_bbox[0]) > (box_w - 60):
                cur_size -= 2
                cur_title_f = ImageFont.truetype("Formula1-Bold_web_0.ttf.ttf", cur_size)
                t_bbox = cur_title_f.getbbox(title_str)
            draw_base.text((box_x + box_w/2 - (t_bbox[2]-t_bbox[0])/2, box_y + 40), title_str, font=cur_title_f, fill=(255, 0, 50, 255))
        except AttributeError: pass
        
        try:
            lap_title_str = "LAP"
            lap_bbox = label_f.getbbox(lap_title_str)
            draw_base.text((box_x + box_w/2 - (lap_bbox[2]-lap_bbox[0])/2, box_y + box_h - 50), lap_title_str, font=label_f, fill=(255, 255, 255, 255))
        except AttributeError: pass

        # --- 2. GRAPH LAYER (Dedicated transparent mask layer) ---
        graph_img = Image.new('RGBA', (int(graph_w), int(graph_h)), (0, 0, 0, 0))
        draw_graph = ImageDraw.Draw(graph_img)

        # Draw connecting horizontal lines for aesthetics
        for pos in range(1, 21):
            draw_graph.line([(0, get_y(pos)), (graph_w, get_y(pos))], fill=(30, 30, 30, 255), width=1)

        for drv, state in current_states.items():
            alpha = state['alpha']
            if alpha == 0: continue
            
            data = d_data[drv]
            team_rgb = hex_to_rgb(data['color'])
            line_color = (*team_rgb, alpha)
            
            pts = []
            for i, t in enumerate(data['times']):
                if t <= state['eval_time']:
                    pts.append((get_x(data['laps'][i]), get_y(data['pos'][i])))
            
            head_x = get_x(state['lap'])
            head_y = get_y(state['pos'])
            pts.append((head_x, head_y))
            
            if len(pts) > 1:
                draw_graph.line(pts, fill=line_color, width=line_w, joint="curve")
                
            draw_graph.ellipse([head_x - dot_r, head_y - dot_r, head_x + dot_r, head_y + dot_r], fill=line_color, outline=(255, 255, 255, alpha), width=max(2, int(line_w*0.4)))
            
            # Gaps, Names, Tires Spacing Fixes
            lbl_x = head_x + dot_r + 20
            try:
                l_bbox = label_f.getbbox(drv)
                lh, lw = l_bbox[3] - l_bbox[1], l_bbox[2] - l_bbox[0]
                draw_graph.text((lbl_x, head_y - lh/2 - 5), drv, font=label_f, fill=(255, 255, 255, alpha))
            except AttributeError:
                lw = 60
                draw_graph.text((lbl_x, head_y - 15), drv, font=label_f, fill=(255, 255, 255, alpha))

            cur_comp_idx = np.searchsorted(data['times'], state['eval_time'], side='right') - 1
            cur_comp_idx = np.clip(cur_comp_idx, 0, len(data['compounds']) - 1)
            tire_hex, tire_letter = get_tire_info(data['compounds'][cur_comp_idx])
            
            # Tire Symbol enlarged
            t_rad = int(dot_r * 1.5)
            t_x = lbl_x + lw + 25
            draw_graph.ellipse([t_x - t_rad, head_y - t_rad, t_x + t_rad, head_y + t_rad], fill=(*hex_to_rgb(tire_hex), alpha))
            try:
                t_bbox = tire_f.getbbox(tire_letter)
                tw, th = t_bbox[2] - t_bbox[0], t_bbox[3] - t_bbox[1]
                draw_graph.text((t_x - tw/2, head_y - th/2 - 4), tire_letter, font=tire_f, fill=(0, 0, 0, alpha))
            except AttributeError: pass
                
            if drv != leader_drv and state['gap'] > 0:
                gap_str = f"+{state['gap']:.1f}"
                # Gap pushed strictly down below driver name
                draw_graph.text((lbl_x, head_y + dot_r + 5), gap_str, font=gap_f, fill=(200, 200, 200, alpha))

        # Merge Layers
        base_img.paste(graph_img, (int(box_x + graph_offset_l), int(box_y + graph_offset_t)), graph_img)

        video.write(cv2.cvtColor(np.array(base_img.convert('RGB')), cv2.COLOR_RGB2BGR))
        if frame_idx % (fps * 2) == 0: print(f"Processed {frame_idx // fps}s / {duration}s")
            
    video.release()
    print("Video rendered successfully!")

if __name__ == "__main__":
    create_race_timelapse(year=2025, gp='Australia', session_type='R', 
                          save_path='output.mp4', duration=30, fps=60, portrait=True)

core           INFO 	Loading data for Australian Grand Prix - Race [v3.9.0.dev2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Loading data for 2025 Australia Session: R...


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Driver 4 completed the race distance 00:00.022000 before the recorded end of the session.
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '1', '63', '12', '23', '18', '27', '16', '81', '44', '10', '22', '31', '87', '30', '5', '14', '55', '7', '6']


Rendering 1800 frames to output.mp4...
Processed 0s / 30s
Processed 2s / 30s
Processed 4s / 30s
Processed 6s / 30s
Processed 8s / 30s
Processed 10s / 30s
Processed 12s / 30s
Processed 14s / 30s
Processed 16s / 30s
Processed 18s / 30s
Processed 20s / 30s
Processed 22s / 30s
Processed 24s / 30s
Processed 26s / 30s
Processed 28s / 30s
Video rendered successfully!


In [5]:
"""
race_replay.py  —  F1 Race Position Timelapse (fast render)
OpenCV + PIL, no matplotlib.

Speed fixes vs v1:
  • All line/circle drawing via cv2 on NumPy arrays  (10-100x faster than PIL)
  • Pre-cache all pixel-coord arrays before render loop  (no per-frame math)
  • PIL used ONLY for text labels, composited once per frame
  • No RGBA overlay allocation per driver — cv2.addWeighted for DNF fade
  • Pre-rendered driver label patches (RGBA numpy patches, blitted per frame)
  • Static background (grid + event name) built once, copied each frame
  • ETA progress display during render
"""

import os
import time
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont
import fastf1
import warnings
warnings.filterwarnings("ignore")

# ─── CONSTANTS ────────────────────────────────────────────────────────────────

TIRE_COLORS_BGR = {          # OpenCV is BGR
    "SOFT":         (50,  50,  220),
    "MEDIUM":       (40,  200, 230),
    "HARD":         (230, 230, 230),
    "INTERMEDIATE": (80,  190,  80),
    "WET":          (220, 120,  60),
    "UNKNOWN":      (150, 150, 150),
}

FONT_BOLD_PATH    = "Formula1-Bold_web_0.ttf.ttf"
FONT_REGULAR_PATH = "Formula1-Regular_web_0.ttf.ttf"

WINDOW_LAPS = 10   # visible lap window
SUB         = 30   # sub-steps per lap  (30 is smooth enough; 60 doubles render time)


# ─── TEAM COLOR LOOKUP ────────────────────────────────────────────────────────

def get_team_color_hex(driver_code: str, year: int, track: str) -> str:
    FALLBACK = {
        "VER": "#3671C6", "PER": "#3671C6",
        "LEC": "#E8002D", "SAI": "#E8002D", "BEA": "#E8002D",
        "NOR": "#FF8000", "PIA": "#FF8000",
        "HAM": "#27F4D2", "RUS": "#27F4D2",
        "ALO": "#358C75", "STR": "#358C75",
        "OCO": "#FF87BC", "GAS": "#FF87BC", "DOO": "#FF87BC",
        "ALB": "#64C4FF", "SAR": "#64C4FF",
        "TSU": "#6692FF", "RIC": "#6692FF", "LAW": "#6692FF", "HAD": "#6692FF",
        "BOT": "#C92D4B", "ZHO": "#C92D4B",
        "MAG": "#B6BABD", "HUL": "#B6BABD",
        "DEV": "#2B4562", "ANT": "#27F4D2",
    }
    return FALLBACK.get(driver_code.upper(), "#FFFFFF")


def hex_to_bgr(h: str):
    h = h.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return (b, g, r)


# ─── FONTS ────────────────────────────────────────────────────────────────────

def load_font(size: int, bold: bool = False) -> ImageFont.FreeTypeFont:
    path = FONT_BOLD_PATH if bold else FONT_REGULAR_PATH
    if os.path.exists(path):
        return ImageFont.truetype(path, size)
    return ImageFont.load_default()


# ─── DATA LOADING ─────────────────────────────────────────────────────────────

def load_race_data(year: int, gp: str, session_type: str):
    cache_dir = os.path.join(os.path.expanduser("~"), ".fastf1_cache")
    os.makedirs(cache_dir, exist_ok=True)
    fastf1.Cache.enable_cache(cache_dir)

    session = fastf1.get_session(year, gp, session_type)
    session.load(laps=True, telemetry=False)

    event_name = f"{year} {gp} \u2014 {'Sprint' if session_type == 'S' else 'Race'}"
    laps = session.laps[["Driver", "LapNumber", "Position", "Compound"]].copy()
    laps = laps.dropna(subset=["LapNumber", "Position"])
    laps["LapNumber"] = laps["LapNumber"].astype(int)
    laps["Position"]  = laps["Position"].astype(int)

    drivers  = sorted(laps["Driver"].unique().tolist())
    max_lap  = int(laps["LapNumber"].max())
    lap_data = {}

    for drv in drivers:
        rows = laps[laps["Driver"] == drv].sort_values("LapNumber")
        records = []
        for _, row in rows.iterrows():
            comp = str(row["Compound"]).upper()
            comp = comp if comp != "NAN" else "UNKNOWN"
            records.append((int(row["LapNumber"]), int(row["Position"]), comp))
        lap_data[drv] = records

    return event_name, drivers, lap_data, max_lap, len(drivers)


# ─── TIMELINE BUILD ───────────────────────────────────────────────────────────

def build_timelines(lap_data: dict, max_lap: int):
    N     = max_lap * SUB
    all_t = np.linspace(1.0, max_lap, N)
    tl    = {}

    for drv, records in lap_data.items():
        if not records:
            continue
        laps_a = np.array([r[0] for r in records], dtype=float)
        pos_a  = np.array([r[1] for r in records], dtype=float)
        comp_l = [r[2] for r in records]

        interp_pos  = np.interp(all_t, laps_a, pos_a).astype(np.float32)

        idx_arr     = np.clip(np.searchsorted(laps_a, all_t, side="right") - 1,
                              0, len(comp_l) - 1)
        interp_comp = [comp_l[i] for i in idx_arr]

        alive = (all_t <= laps_a[-1] + 0.5)

        tl[drv] = {"pos": interp_pos, "compound": interp_comp, "alive": alive}

    return tl


# ─── PRE-COMPUTE PIXEL Y COORDS (fixed, scroll-independent) ──────────────────

def precompute_coords(tl: dict, max_lap: int, total_drivers: int,
                      W: int, H: int, ML: int, MR: int, MT: int, MB: int):
    """
    x_rel[i] = fractional position along plot_w for timeline step i  (0..plot_w)
    y[i]     = absolute pixel y for driver's position at step i
    """
    plot_w = W - ML - MR
    plot_h = H - MT  - MB
    N      = max_lap * SUB
    coords = {}

    x_rel_all = np.linspace(0.0, float(plot_w), N, dtype=np.float32)

    for drv, d in tl.items():
        y_arr = (MT + (d["pos"] - 1.0) / max(total_drivers - 1, 1) * plot_h).astype(np.float32)
        coords[drv] = {"x_rel": x_rel_all, "y": y_arr}   # x_rel shared ref is fine (read-only)

    return coords, x_rel_all


# ─── PRE-RENDER DRIVER LABEL PATCHES ─────────────────────────────────────────

def build_label_cache(drivers, fonts, year, gp):
    cache = {}
    for drv in drivers:
        color_bgr = hex_to_bgr(get_team_color_hex(drv, year, gp))
        color_rgb = color_bgr[::-1]
        f   = fonts["driver"]
        bb  = f.getbbox(drv)
        tw, th = bb[2] - bb[0], bb[3] - bb[1]
        pad = 7
        pw, ph = tw + pad * 2, th + pad * 2

        pil = Image.new("RGBA", (pw, ph), (20, 20, 20, 200))
        ImageDraw.Draw(pil).text((pad - bb[0], pad - bb[1]),
                                  drv, font=f, fill=(*color_rgb, 255))
        cache[drv] = {
            "patch":     np.array(pil, dtype=np.uint8),  # RGBA
            "w": pw, "h": ph,
            "color_bgr": color_bgr,
        }
    return cache


# ─── ALPHA COMPOSITE ──────────────────────────────────────────────────────────

def blit_rgba(canvas, patch, x, y):
    """Composite RGBA patch onto BGR canvas."""
    H_c, W_c = canvas.shape[:2]
    ph, pw    = patch.shape[:2]
    cx0 = max(0, x);    cy0 = max(0, y)
    cx1 = min(W_c, x + pw); cy1 = min(H_c, y + ph)
    px0 = cx0 - x;     py0 = cy0 - y
    px1 = px0 + (cx1 - cx0); py1 = py0 + (cy1 - cy0)
    if cx1 <= cx0 or cy1 <= cy0:
        return
    roi   = canvas[cy0:cy1, cx0:cx1].astype(np.float32)
    src   = patch[py0:py1, px0:px1]
    alpha = src[:, :, 3:4].astype(np.float32) / 255.0
    rgb   = src[:, :, :3][:, :, ::-1].astype(np.float32)
    canvas[cy0:cy1, cx0:cx1] = np.clip(roi * (1 - alpha) + rgb * alpha, 0, 255).astype(np.uint8)


def draw_text_pill(canvas, text, x, y, font_pil, color_rgb, pad=6, bg=(20, 20, 20), bg_a=200):
    bb  = font_pil.getbbox(text)
    tw, th = bb[2] - bb[0], bb[3] - bb[1]
    pw, ph = tw + pad * 2, th + pad * 2
    pil = Image.new("RGBA", (pw, ph), (*bg, bg_a))
    ImageDraw.Draw(pil).text((pad - bb[0], pad - bb[1]), text,
                              font=font_pil, fill=(*color_rgb, 255))
    blit_rgba(canvas, np.array(pil, dtype=np.uint8), x, y)


# ─── STATIC BACKGROUND ────────────────────────────────────────────────────────

def build_static_bg(W, H, total_drivers, event_name,
                    ML, MR, MT, MB, fonts):
    bg = np.zeros((H, W, 3), dtype=np.uint8)
    plot_w = W - ML - MR
    plot_h = H - MT  - MB

    # Horizontal grid lines (cv2 — fast)
    for p in range(1, total_drivers + 1):
        y = int(MT + (p - 1) / max(total_drivers - 1, 1) * plot_h)
        cv2.line(bg, (ML, y), (ML + plot_w, y), (35, 35, 35), 1)

    # P labels + event name (PIL — text only, once)
    pil = Image.fromarray(cv2.cvtColor(bg, cv2.COLOR_BGR2RGB))
    drw = ImageDraw.Draw(pil)

    for p in range(1, total_drivers + 1):
        y = int(MT + (p - 1) / max(total_drivers - 1, 1) * plot_h)
        drw.text((ML - 45, y - 10), f"P{p}", font=fonts["small"], fill=(90, 90, 90))

    eb  = fonts["event"].getbbox(event_name)
    ew  = eb[2] - eb[0]
    ex  = (W - ew) // 2
    ey  = 14
    drw.rounded_rectangle([ex - 16, ey - 4, ex + ew + 16, ey + 42], radius=6, fill=(20, 20, 20))
    drw.text((ex, ey), event_name, font=fonts["event"], fill=(220, 220, 220))

    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


# ─── RENDER FRAME ─────────────────────────────────────────────────────────────

def render_frame(vf, total_video_frames, timeline_total,
                 tl, drivers, max_lap, total_drivers,
                 coords, x_rel_all, label_cache,
                 static_bg, fonts,
                 W, H, ML, MR, MT, MB,
                 year, gp):

    plot_w = W - ML - MR
    plot_h = H - MT  - MB

    t_frac            = vf / max(total_video_frames - 1, 1)
    current_lap_float = 1.0 + t_frac * (max_lap - 1)
    current_lap_int   = max(1, min(max_lap, round(current_lap_float)))

    # Scrolling window
    half       = WINDOW_LAPS / 2.0
    view_start = current_lap_float - half
    view_end   = current_lap_float + half
    if view_start < 1:
        view_start, view_end = 1.0, float(WINDOW_LAPS + 1)
    if view_end > max_lap:
        view_end = float(max_lap)
        view_start = view_end - WINDOW_LAPS
    view_start = max(1.0, view_start)

    # x_rel bounds for the visible window
    xs_rel = (view_start - 1.0) / (max_lap - 1) * plot_w
    xe_rel = (view_end   - 1.0) / (max_lap - 1) * plot_w
    vis_w  = max(xe_rel - xs_rel, 1.0)

    def xrel_to_cx(xr):
        return int(ML + (xr - xs_rel) / vis_w * plot_w)

    frame = static_bg.copy()

    # Vertical lap lines + labels (cv2 for lines, PIL for text)
    lap_labels_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    lap_lbl_drw    = ImageDraw.Draw(lap_labels_pil)
    for lap in range(max(1, int(view_start) - 1), min(max_lap, int(view_end)) + 2):
        xr = (lap - 1) / (max_lap - 1) * plot_w
        cx = xrel_to_cx(xr)
        if ML <= cx <= ML + plot_w:
            cv2.line(frame, (cx, MT), (cx, MT + plot_h), (28, 28, 28), 1)
            lap_lbl_drw.text((cx - 8, MT + plot_h + 6), str(lap),
                             font=fonts["small"], fill=(70, 70, 70))
    # Blit lap labels back
    frame = cv2.cvtColor(np.array(lap_labels_pil), cv2.COLOR_RGB2BGR)

    tf_now = min(int(t_frac * (timeline_total - 1)), timeline_total - 1)

    # ── Trails ──────────────────────────────────────────────────────────────
    for drv in drivers:
        if drv not in tl or drv not in coords:
            continue
        d     = tl[drv]
        c     = coords[drv]
        color = label_cache[drv]["color_bgr"]

        x_rel = c["x_rel"]
        y_arr = c["y"]

        # Index range: [0, tf_now] clipped to visible x
        i0 = max(0, int((xs_rel / plot_w) * (len(x_rel) - 1)) - 1)
        i1 = min(tf_now + 1, int((xe_rel / plot_w) * (len(x_rel) - 1)) + 2)
        if i1 <= i0:
            continue

        xi = x_rel[i0:i1]
        yi = y_arr[i0:i1]
        cx_arr = (ML + (xi - xs_rel) / vis_w * plot_w).astype(np.int32)
        cy_arr = yi.astype(np.int32)

        alive_mask = d["alive"][i0:i1]
        in_view    = (cx_arr >= ML - 2) & (cx_arr <= ML + plot_w + 2)
        valid      = alive_mask & in_view

        if valid.sum() < 2:
            continue

        pts = np.stack([cx_arr[valid], cy_arr[valid]], axis=1).reshape(-1, 1, 2)
        alive_now = bool(d["alive"][min(tf_now, len(d["alive"]) - 1)])

        if alive_now:
            cv2.polylines(frame, [pts], False, color, 3, cv2.LINE_AA)
        else:
            overlay = frame.copy()
            cv2.polylines(overlay, [pts], False, color, 1, cv2.LINE_AA)
            cv2.addWeighted(overlay, 0.25, frame, 0.75, 0, frame)

    # ── Dots + labels ────────────────────────────────────────────────────────
    cur_xrel = (current_lap_float - 1.0) / (max_lap - 1) * plot_w
    dot_cx   = xrel_to_cx(cur_xrel)

    current_positions = {}
    for drv in drivers:
        if drv not in tl:
            continue
        d  = tl[drv]
        fi = min(tf_now, len(d["pos"]) - 1)
        if d["alive"][fi]:
            current_positions[drv] = float(d["pos"][fi])

    leader_pos = min(current_positions.values()) if current_positions else 1.0

    for drv in sorted(current_positions, key=lambda d: current_positions[d]):
        d        = tl[drv]
        fi       = min(tf_now, len(d["pos"]) - 1)
        cur_comp = d["compound"][fi]
        color    = label_cache[drv]["color_bgr"]
        dot_cy   = int(coords[drv]["y"][fi])

        cv2.circle(frame, (dot_cx, dot_cy), 9, color, -1)
        cv2.circle(frame, (dot_cx, dot_cy), 9, (255, 255, 255), 1, cv2.LINE_AA)

        tire_bgr = TIRE_COLORS_BGR.get(cur_comp, TIRE_COLORS_BGR["UNKNOWN"])
        cv2.circle(frame, (dot_cx + 22, dot_cy), 6, tire_bgr, -1)
        cv2.circle(frame, (dot_cx + 22, dot_cy), 6, (50, 50, 50), 1, cv2.LINE_AA)

        lc = label_cache[drv]
        blit_rgba(frame, lc["patch"], dot_cx - lc["w"] // 2, dot_cy - lc["h"] - 12)

        my_pos = current_positions[drv]
        if my_pos > leader_pos + 0.4:
            draw_text_pill(frame, f"+{my_pos - leader_pos:.1f}s",
                           dot_cx - 24, dot_cy + 14,
                           fonts["small"], (180, 180, 180))

    # ── LAP counter ─────────────────────────────────────────────────────────
    lap_text = f"LAP {current_lap_int} / {max_lap}"
    lb = fonts["lap"].getbbox(lap_text)
    lw = lb[2] - lb[0]
    draw_text_pill(frame, lap_text,
                   W - MR - lw - 28, H - MB + 20,
                   fonts["lap"], (255, 255, 255), pad=10)

    return frame


# ─── MAIN ENTRY ───────────────────────────────────────────────────────────────

def create_race_timelapse(
    year: int         = 2025,
    gp: str           = "Qatar Grand Prix",
    session_type: str = "S",
    save_path: str    = "output.mp4",
    duration: int     = 30,
    fps: int          = 60,
    portrait: bool    = True,
):
    W, H = (1080, 1920) if portrait else (1920, 1080)
    ML, MR, MT, MB = int(W*.08), int(W*.08), int(H*.10), int(H*.12)

    print(f"[race_replay] Loading {year} {gp} ({session_type})…")
    event_name, drivers, lap_data, max_lap, total_drivers = load_race_data(year, gp, session_type)
    print(f"[race_replay] {len(drivers)} drivers, {max_lap} laps — {event_name}")

    tl = build_timelines(lap_data, max_lap)

    base  = H if portrait else W
    fonts = {
        "event":  load_font(int(base * 0.022), bold=True),
        "driver": load_font(int(base * 0.018), bold=True),
        "small":  load_font(int(base * 0.013), bold=False),
        "lap":    load_font(int(base * 0.030), bold=True),
    }

    coords, x_rel_all = precompute_coords(tl, max_lap, total_drivers, W, H, ML, MR, MT, MB)
    label_cache        = build_label_cache(drivers, fonts, year, gp)
    static_bg          = build_static_bg(W, H, total_drivers, event_name, ML, MR, MT, MB, fonts)

    total_frames   = duration * fps
    timeline_total = max_lap * SUB

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out    = cv2.VideoWriter(save_path, fourcc, fps, (W, H))

    print(f"[race_replay] Rendering {total_frames} frames ({duration}s @ {fps}fps)…")
    t0 = time.time()

    for vf in range(total_frames):
        frame = render_frame(
            vf, total_frames, timeline_total,
            tl, drivers, max_lap, total_drivers,
            coords, x_rel_all, label_cache,
            static_bg, fonts,
            W, H, ML, MR, MT, MB,
            year, gp,
        )
        out.write(frame)

        if vf % 30 == 0:
            elapsed = time.time() - t0
            rate    = (vf + 1) / max(elapsed, 1e-3)
            eta     = (total_frames - vf - 1) / max(rate, 1e-3)
            print(f"  {vf+1:>5}/{total_frames}  {rate:5.1f} fps  ETA {eta:5.0f}s    ", end="\r")

    out.release()
    print(f"\n[race_replay] \u2713 Done in {time.time()-t0:.1f}s \u2192 {save_path}")


# ─── CLI ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    create_race_timelapse(
        year=2025,
        gp="Qatar Grand Prix",
        session_type="S",
        save_path="output.mp4",
        duration=30,
        fps=60,
        portrait=True,
    )

events      WARNING 	Correcting user input 'Qatar Grand Prix' to 'Qatar Grand Prix'
core           INFO 	Loading data for Qatar Grand Prix - Sprint [v3.9.0.dev2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '4'
core        WARNING 	Fixed incorrect tyre stint information for driver '1'
core        WARNING 	Fixed incorrect tyre stint information for driver '22'
core        WARNING 	Fixed incorrect tyre stint information for driver '12'
core        WARNING 	Fixed incorrect tyre stint information for driver '14'


[race_replay] Loading 2025 Qatar Grand Prix (S)…


core        WARNING 	Fixed incorrect tyre stint information for driver '6'
core        WARNING 	Fixed incorrect tyre stint information for driver '23'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Fixed incorrect tyre stint information for driver '16'
core        WARNING 	Fixed incorrect tyre stint information for driver '10'
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '63', '4', '1', '22', '12', '14', '55', '6', '23', '5', '87', '16', '30', '31', '27', '44', '10', '18', '43']


[race_replay] 20 drivers, 19 laps — 2025 Qatar Grand Prix — Sprint
[race_replay] Rendering 1800 frames (30s @ 60fps)…
   1771/1800   30.8 fps  ETA     1s    
[race_replay] ✓ Done in 58.6s → output.mp4
